# Constant Acceleration MPC Testing

In [ ]:
%matplotlib ipympl

import time
import functools
import jax
import jax.numpy as jnp
import numpy as np
import tqdm
import scipy.optimize as sci_opt

import matplotlib.pyplot as plt

import exp_mpc.stewart_min.viz as viz
import exp_mpc.stewart_min.spec as spec
import exp_mpc.stewart_min.const as const
import exp_mpc.stewart_min.opt as opt
import exp_mpc.stewart_min.quartic_cost as quartic_cost

jax.config.update("jax_enable_x64", True)


In [ ]:
# general setup

T = 10.0  # s
num_steps = int(T / const.dt)
n = 400  # horizon

acc_ref = jnp.array([1.0, 0.0, 0.0]) + const.gravity
acc_ref = jnp.tile(A=acc_ref, reps=(n, 1))
omega_ref = jnp.array([0.0, 0.0, 0.0])
omega_ref = jnp.tile(A=omega_ref, reps=(n, 1))

In [ ]:
# cost setup

weights = opt.Weights(
    acc=jnp.array([1e2, 1e2, 1.0]),
    alpha_acc=jnp.array([4.0]),
)

margins = [0.2, 0.1]
sizes = [1.0, 2**3, 2**8]

leg_cost = quartic_cost.QuarticCost.from_bounds(
    margins=margins,
    sizes=sizes,
    low=const.leg_min**2,
    high=const.leg_max**2,
)
joint_angle_cost = quartic_cost.QuarticCost.from_bounds(
    margins=margins,
    sizes=sizes,
    low=-const.joint_max_angle,
    high=const.joint_max_angle,
)
yaw_cost = quartic_cost.QuarticCost.from_bounds(
    margins=margins,
    sizes=sizes,
    low=-const.max_yaw,
    high=const.max_yaw,
)


In [ ]:
# train setup
acados_get_cost = functools.partial(
    opt.get_cost,
    weights=weights,
    acc_ref=acc_ref,
    omega_ref=omega_ref,
    leg_cost=leg_cost,
    joint_angle_cost=joint_angle_cost,
    yaw_cost=yaw_cost,
    # state0=state0,
)
    
def train_step(
    state0: jax.Array, last_control: jax.Array
) -> tuple[jax.Array, jax.Array, spec.TableSol, sci_opt.OptimizeResult, float]:
    cost, cost_jac, _ = acados_get_cost(state0=state0)
    t0 = time.time()
    res = sci_opt.minimize(
        fun=cost,
        x0=np.array(last_control),
        method="L-BFGS-B",
        jac=cost_jac,
        options={
            "maxiter": 16,
            "maxls": 8,
        },
    )
    t1 = time.time()
    t_tot = t1 - t0
    control = opt.Control.from_flat(jnp.array(res.x))
    state = control.get_state(state0)
    table_sol = spec.TableSol(
        x=np.array(jnp.column_stack(list(state))),
        u=np.array(jnp.column_stack(list(control))),
        stats=spec.TableStats(
            time=t_tot,
            status=res.status,
            cost=res.fun,
        )
    )
    return state[1].flatten(), control.flatten(), table_sol, res, t_tot


In [ ]:
# run

state0 = jnp.zeros(12, dtype=float)
last_control = jnp.zeros(6 * n, dtype=float)
times = []
sol_list = []
res_list = []

for _ in tqdm.tqdm(range(num_steps)):
    state0, last_control, sol, res, t_tot = train_step(state0, last_control)
    sol_list.append(sol)
    res_list.append(res)
    times.append(t_tot)


In [ ]:
def max_abs(arr):
    return jnp.max(jnp.abs(arr))

def max_abs_arg(arr):
    return jnp.argmax(jnp.abs(arr))

arr = opt.Control.from_flat(sol_list[-1].u.flatten()).z
max_abs(arr), max_abs_arg(arr), arr

In [ ]:
plt.close("all")

In [ ]:
anim_fig = viz.animate_trajectory(
    trajectory=sol_list,
    sim_rate=1.0,
    fps=30,
)

In [ ]:
# # sol_list_end = []
# extra_steps = 0

sol_list_end = viz.split_tablesol(sol_list[-1])
extra_steps = sol_list[-1].u.shape[0] - 1

In [ ]:
acados_human_fig = viz.plot_human_trajectory(
    trajectory=sol_list + sol_list_end,
    references={
        "xyz-acceleration": np.tile(
            A=acc_ref[0], reps=(num_steps + extra_steps, 1)
        ),
        "angular-velocity": np.tile(
            A=omega_ref[0], reps=(num_steps + extra_steps, 1)
        ),
    },
)

In [ ]:
acados_table_fig = viz.plot_cartesian_table_trajectory(
    trajectory=sol_list + sol_list_end,
)

In [ ]:
acados_actuator_fig = viz.plot_actuator_trajectory(
    trajectory=sol_list + sol_list_end,
)